# Modélisation – H3b : logement et besoins des ménages
**Contributeurs :** Cécile · **Date :** 04/11/2026

Notebook construit à partir du template commun de l'équipe. Il passe par `commun.py` pour tout ce qui doit être identique : la table, le masque des lignes, les fenêtres, la naïve, `evaluer` et `tester`.

**Question :** le parc de logements donne-t-il un prix attendu stable, qui aide surtout dans les petits territoires ? Les revenus disent-ils la même chose que le parc ?

## 0. Paramètres de l'hypothèse
Les variables sont rangées par bloc, dans l'ordre où elles entrent dans les modèles :
- **parc** : les 9 indicateurs du parc de logements (recensement 2022) ;
- **vacance** : le taux de vacance de LOVAC (celui du recensement est déjà dans le parc) ;
- **ménages** : 4 indicateurs de la structure des ménages (recensements 2019 à 2022), ajoutés à la fiche depuis leur entrée dans la table ;
- **revenus** : le niveau de vie médian (Filosofi) ;
- **densité** : en variante seulement.

La taille des ménages n'apparaît plus : l'audit complémentaire (version 2) l'a retirée de la table, redondante avec la part de personnes seules.

`VARIABLES_LOG` et `VARIABLES_LOG1P` : les variables asymétriques passées en log (règle de l'audit : asymétrie > 1), pour les seuls modèles qui en ont besoin.

In [1]:
HYPOTHESE = "H3b"

PARC = ["taux_vacance", "part_rsec", "part_apparts", "log_nb_logements", "part_moins40m2",
        "part_avant1946", "part_2006_2019", "anciennete_moyenne", "part_sans_voiture"]
VACANCE = ["taux_vacance_complete"]
MENAGES = ["taux_personnes_seules_pct_men", "part_familles_monoparentales_pct_fam",
           "part_familles_nombreuses_pct_fam", "part_couples_sans_enfants_pct_fam"]
REVENUS = ["niveau_vie_median"]
DENSITE = ["densite_population"]

VARIABLES_BLOC = PARC + VACANCE + MENAGES + REVENUS      # utilisé par le constat rapide (section 2)

# Variables asymétriques (audit complémentaire, version 2 : asymétrie > 1) : log, ou log(1 + x) si elles contiennent des zéros
LOG = ["nb_ventes_n1", "densite_population", "niveau_vie_median",
       "taux3_vols_sans_violence", "taux3_vols_vehicules",
       "taux3_violences_sexuelles", "taux3_cambriolages", "taux3_violences_intrafamiliales",
       "taux_equip_commerces", "taux_equip_services", "taux_equip_sante", "taux_equip_sports",
       "part_sans_voiture", "part_moins40m2", "part_rsec", "part_apparts",
       "part_familles_monoparentales_pct_fam"]
LOG1P = ["taux3_vols_violents", "taux3_stupefiants", "taux_equip_tourisme", "taux_equip_transports"]

# Variables de H3b seulement
VARIABLES_LOG = [v for v in LOG if v in VARIABLES_BLOC + DENSITE]
VARIABLES_LOG1P = [v for v in LOG1P if v in VARIABLES_BLOC + DENSITE]
print("log :", VARIABLES_LOG, "| log(1 + x) :", VARIABLES_LOG1P)                                                    

log : ['densite_population', 'niveau_vie_median', 'part_sans_voiture', 'part_moins40m2', 'part_rsec', 'part_apparts', 'part_familles_monoparentales_pct_fam'] | log(1 + x) : []


## 1. Import et chargement
`charger_table()` retire une seule fois les lignes à case vide (masque commun) : tous les modèles et la naïve sont jugés sur les mêmes lignes. `fenetres()` donne une fenêtre par année jugée (2023 et 2024), avec l'apprentissage sur les années d'avant.

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import TransformedTargetRegressor
from gabarit_boosting import BoostingEvolution, prevoir_boosting
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from copy import deepcopy

import commun
from commun import CIBLE, charger_table, fenetres, plis_temporels, evaluer, tester
from gabarit_hierarchique import ModeleHierarchique

table = charger_table()
FENETRES = fenetres(table)

absentes = sorted(set(VARIABLES_BLOC + VARIABLES_LOG) - set(table.columns))
assert not absentes, f"Variables absentes de la table : {absentes}"
print("Table :", table.shape)
for a, (app, ev) in FENETRES.items():
    print(f"Fenêtre {a} : apprentissage {sorted(app['annee'].unique())} ({len(app)} lignes) | jugée {a} ({len(ev)} lignes)")

Table : (23489, 39)
Fenêtre 2023 : apprentissage [np.int64(2020), np.int64(2021), np.int64(2022)] (7987 lignes) | jugée 2023 (2641 lignes)
Fenêtre 2024 : apprentissage [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] (10628 lignes) | jugée 2024 (2599 lignes)


## 2. Constat rapide
Distribution des variables du bloc et lien avec le prix (rho de Spearman), sur les années d'apprentissage de la première fenêtre seulement : l'année jugée n'est jamais regardée avant la modélisation.

In [3]:
app_constat = FENETRES[commun.ANNEES_JUGEES[0]][0]
print(app_constat[VARIABLES_BLOC].describe().round(2).to_string())

liens = []
for type_bien, e in app_constat.groupby("type_bien"):
    for v in VARIABLES_BLOC:
        rho, p = stats.spearmanr(e[v], e[CIBLE])
        liens.append({"type de bien": type_bien, "variable": v, "rho": rho, "p-value": p})
pd.DataFrame(liens).pivot(index="variable", columns="type de bien", values="rho").round(2)

       taux_vacance  part_rsec  part_apparts  log_nb_logements  part_moins40m2  part_avant1946  part_2006_2019  anciennete_moyenne  part_sans_voiture  taux_vacance_complete  taux_personnes_seules_pct_men  part_familles_monoparentales_pct_fam  part_familles_nombreuses_pct_fam  part_couples_sans_enfants_pct_fam  niveau_vie_median
count       7987.00    7987.00       7987.00           7987.00         7987.00         7987.00         7987.00             7987.00            7987.00                7987.00                        7987.00                               7987.00                           7987.00                            7987.00            7987.00
mean           8.18      12.92         26.25              4.10            6.01           23.18           15.66               16.66              10.89                  11.44                          34.85                                 14.27                              7.57                              46.64           23142.30
std       

type de bien,appartement,maison
variable,,
anciennete_moyenne,-0.46,-0.71
log_nb_logements,0.03,0.28
niveau_vie_median,0.62,0.66
part_2006_2019,0.35,0.62
part_apparts,0.32,0.49
part_avant1946,-0.46,-0.76
part_couples_sans_enfants_pct_fam,-0.12,-0.42
part_familles_monoparentales_pct_fam,-0.02,0.16
part_familles_nombreuses_pct_fam,-0.24,0.01


## 3. Fonctions du notebook
`lancer` fait tourner un modèle sur toutes les fenêtres, un modèle par type de bien, et ajoute ses lignes aux tableaux communs (`evaluer`, `tester`).
- **Pipeline** : mise à l'échelle puis modèle ; les paramètres de la mise à l'échelle sont appris sur l'apprentissage seulement.
- **`log=True`** : les `VARIABLES_LOG` et la cible passent en log ; la prévision est remise en €/m² avant `evaluer`. À réserver aux modèles qui en ont besoin (régressions, Ridge / Lasso / Elastic Net), jamais aux arbres.
- **`grille`** : réglage par `GridSearchCV` sur les plis temporels de `commun.py` (une année de validation par pli, jamais l'année jugée).
- **Modèle hiérarchique** (`ModeleHierarchique` de `gabarit_hierarchique.py`) : pas de pipeline (il met lui-même les variables à l'échelle) ni de grille ; il lit le sous-bassin dans l'index, que `lancer` conserve. Avec `log=True`, il apprend sur le log du prix et ses prévisions sont remises en €/m².
- **Gradient boosting** (`BoostingEvolution` de `gabarit_boosting.py`, gabarit commun P3) : pas de pipeline ni de log. `lancer` appelle `prevoir_boosting`, qui règle le modèle avec la grille commune (`GRILLE_REFERENCE`) sur les plis temporels. Le modèle apprend l'évolution log(prix / prix N-1) et rend un prix en €/m² (prix N-1 × évolution prévue) : la cible jugée reste le prix. `grille={}` : pas de réglage.

In [4]:
resultats, tests, ajustes, previsions = [], [], {}, {}


def en_log(X):
    """Variables asymétriques en log ; log(1 + x) pour celles qui ont des zéros (audit complémentaire)."""
    X = X.copy()
    cols = [c for c in VARIABLES_LOG if c in X.columns]
    cols1p = [c for c in VARIABLES_LOG1P if c in X.columns]
    assert (X[cols] > 0).all().all(), "valeur nulle ou négative dans une variable de VARIABLES_LOG : la passer dans LOG1P"
    X[cols] = np.log(X[cols])
    X[cols1p] = np.log1p(X[cols1p])
    return X


class Hierarchique(RegressorMixin, BaseEstimator):
    """Enveloppe scikit-learn du modèle hiérarchique : log de la cible si demandé, prévisions en €/m²."""
    def __init__(self, modele=None, log=False):
        self.modele = modele
        self.log = log

    def fit(self, X, y):
        self.modele_ = deepcopy(self.modele).fit(X, np.log(y) if self.log else y)
        return self

    def predict(self, X):
        p = np.asarray(self.modele_.predict(X))
        return np.exp(p) if self.log else p


def lancer(nom, modele, variables, log=False, grille=None):
    """Ajuste un modèle par fenêtre et par type de bien, puis evaluer et tester ; affiche le résultat."""
    nom = f"{HYPOTHESE} – {nom}"
    # Relancer une cellule remplace les résultats du modèle au lieu de les ajouter une seconde fois
    resultats[:] = [r for r in resultats if r["modèle"] != nom]
    tests[:] = [t for t in tests if t["modèle"] != nom]
    for annee, (app, ev) in FENETRES.items():
        if isinstance(modele, BoostingEvolution):
            # Gabarit commun P3 : réglage, un modèle par type de bien et prévision en €/m² faits par prevoir_boosting
            assert not log, "pas de log pour le boosting : le gabarit transforme lui-même la cible"
            prevision, modeles = prevoir_boosting(app, ev, variables, cible=modele.cible, grille=grille)
            for type_bien, m in modeles.items():
                idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
                ajustes[(nom, annee, type_bien)] = (m, ev.loc[idx, variables], ev.loc[idx, CIBLE])
        else:
            prevision = pd.Series(np.nan, index=ev.index)
            for type_bien, e in app.groupby("type_bien"):
                idx = ev.index[ev.index.get_level_values("type_bien") == type_bien]
                X_app, X_ev = e[variables], ev.loc[idx, variables]
                if isinstance(modele, ModeleHierarchique):
                    assert not grille, "pas de grille pour le modèle hiérarchique"
                    if log:
                        X_app, X_ev = en_log(X_app), en_log(X_ev)
                    m = Hierarchique(modele, log).fit(X_app, e[CIBLE])
                    prevision.loc[idx] = m.predict(X_ev)
                    ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
                    continue
                m = Pipeline([("echelle", StandardScaler()), ("modele", clone(modele))])
                prefixe = "modele__"
                if log:
                    X_app, X_ev = en_log(X_app), en_log(X_ev)
                    m = TransformedTargetRegressor(regressor=m, func=np.log, inverse_func=np.exp)
                    prefixe = "regressor__modele__"
                if grille:
                    # plis recalculés sur e (positions dans e, même ordre que X_app)
                    m = GridSearchCV(m, {prefixe + k: v for k, v in grille.items()},
                                     cv=plis_temporels(e), scoring="neg_mean_absolute_error")
                m.fit(X_app, e[CIBLE])
                prevision.loc[idx] = m.predict(X_ev)
                ajustes[(nom, annee, type_bien)] = (m, X_ev, ev.loc[idx, CIBLE])
        assert prevision.notna().all(), f"{nom} : prévisions manquantes"
        previsions[(nom, annee)] = prevision          # gardée pour comparer les modèles entre eux (section 5)
        evaluer(nom, ev, prevision, resultats)
        tester(nom, ev, prevision, tests)

    # Résultat du modèle, affiché tout de suite : erreur du modèle, erreur de la naïve, gain
    res = pd.DataFrame(resultats)
    res = res[res["modèle"] == nom][["type de bien", "année jugée", "MAE (€/m²)", "MAE naïve (€/m²)",
                                     "gain sur la naïve (%)", "bassins où le modèle gagne (%)"]]
    print(nom, f"({len(variables)} variables)")
    print(res.round(1).to_string(index=False))
    return nom

## 4. Modèles
Du plus simple au plus complexe ; chaque étape ajoute un bloc à la précédente. Tous les modèles sont gardés, y compris ceux qui perdent.

### La fonction `lancer` (section 3)
Chaque modèle est lancé en une ligne : `lancer(nom, modèle, variables, log=..., grille=...)`.
- `nom` : le nom du modèle dans le tableau commun (le préfixe « H3b » est ajouté automatiquement) ;
- `modèle` : le modèle scikit-learn à utiliser (régression linéaire, Lasso, gradient boosting) ;
- `variables` : les colonnes de la table données au modèle ;
- `log=True` : la cible, `prix_n1` et les variables asymétriques passent en log ;
- `grille` : les valeurs d'un réglage à tester (par exemple `alpha` pour le Lasso).

Ce que fait `lancer`, pour chacune des deux années jugées (2023, puis 2024) et pour chaque type de bien (maisons, appartements) :
1. elle prend les années d'apprentissage, avant l'année jugée (2020-2022 pour juger 2023, 2020-2023 pour juger 2024) ;
2. elle met les variables à l'échelle et entraîne le modèle sur ces années ; avec une grille, elle choisit d'abord le meilleur réglage, chaque année d'apprentissage servant tour à tour de validation ;
3. elle prévoit le prix de l'année jugée, remis en €/m² si le modèle a appris en log ;
4. elle compare cette prévision au prix réel et à la naïve (prix N = prix N-1), avec `evaluer` et `tester` de `commun.py`, et range les résultats dans les tableaux communs.

Elle affiche ensuite, pour chaque année jugée et chaque type de bien :
- **MAE** : l'erreur moyenne du modèle, en €/m² ;
- **MAE naïve** : l'erreur moyenne de la naïve, sur les mêmes sous-bassins ;
- **gain sur la naïve** : de combien le modèle réduit l'erreur de la naïve, en % (négatif : le modèle fait moins bien) ;
- **bassins où le modèle gagne** : la part des sous-bassins où le modèle se trompe moins que la naïve.

Les modèles entraînés sont gardés dans `ajustes`, pour l'interprétation (section 6).

### Le log
Les régressions (et le Lasso) sont faites en log : la cible, `prix_n1` et les variables asymétriques passent en log, les prévisions sont remises en €/m² avant `evaluer`. Avec `prix_n1` en log, le modèle compare log(prix) à log(prix N-1) : il raisonne en évolution relative. La référence est elle aussi en log, pour une comparaison à égalité. Le gradient boosting n'a pas besoin du log : le gabarit transforme lui-même la cible (section 4.7).

In [5]:
# prix_n1 passe en log avec la cible dans les régressions (choix H3b)
VARIABLES_LOG = [v for v in LOG if v in VARIABLES_BLOC + DENSITE] + ["prix_n1"]

# Blocs successifs : chaque étape ajoute un bloc à la précédente
REFERENCE_VARS = ["prix_n1"]
BLOC_PARC_VACANCE = REFERENCE_VARS + PARC + VACANCE
BLOC_MENAGES = BLOC_PARC_VACANCE + MENAGES
BLOC_COMPLET = BLOC_MENAGES + REVENUS
for nom, b in [("parc + vacance", BLOC_PARC_VACANCE), ("+ ménages", BLOC_MENAGES), ("complet", BLOC_COMPLET)]:
    print(f"{nom} : {len(b)} variables")
print("log :", VARIABLES_LOG)

parc + vacance : 11 variables
+ ménages : 15 variables
complet : 16 variables
log : ['densite_population', 'niveau_vie_median', 'part_sans_voiture', 'part_moins40m2', 'part_rsec', 'part_apparts', 'part_familles_monoparentales_pct_fam', 'prix_n1']


### 4.1 Référence : prix N-1 seul
La naïve (prix N = prix N-1) est déjà dans chaque ligne du tableau (« MAE naïve »). Le modèle « prix N-1 seul » est la référence que le bloc doit battre.

In [6]:
REFERENCE = lancer("Régression linéaire – prix N-1 seul", LinearRegression(), REFERENCE_VARS, log=True)

H3b – Régression linéaire – prix N-1 seul (1 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       185.5             166.3                  -11.5                            45.6
      maison         2023       170.0             111.7                  -52.2                            31.0
 appartement         2024       221.5             148.0                  -49.7                            27.0
      maison         2024       214.9             129.2                  -66.4                            18.2


### 4.2 Régression linéaire : + parc et vacance

In [7]:
lancer("Régression linéaire – prix N-1 + parc + vacance", LinearRegression(), BLOC_PARC_VACANCE, log=True)

H3b – Régression linéaire – prix N-1 + parc + vacance (11 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       171.3             166.3                   -3.0                            48.1
      maison         2023       159.8             111.7                  -43.1                            31.7
 appartement         2024       207.6             148.0                  -40.3                            29.4
      maison         2024       200.6             129.2                  -55.2                            20.2


'H3b – Régression linéaire – prix N-1 + parc + vacance'

### 4.3 Régression linéaire : + ménages

In [8]:
lancer("Régression linéaire – prix N-1 + parc + vacance + ménages", LinearRegression(), BLOC_MENAGES, log=True)

H3b – Régression linéaire – prix N-1 + parc + vacance + ménages (15 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       175.2             166.3                   -5.3                            47.3
      maison         2023       166.0             111.7                  -48.6                            30.7
 appartement         2024       211.3             148.0                  -42.8                            28.4
      maison         2024       204.7             129.2                  -58.5                            20.0


'H3b – Régression linéaire – prix N-1 + parc + vacance + ménages'

### 4.4 Régression linéaire : + revenus (bloc complet)
Les revenus disent-ils la même chose que le parc ? Si le gain s'arrête à l'étape précédente, oui.

In [9]:
COMPLET = lancer("Régression linéaire – bloc complet", LinearRegression(), BLOC_COMPLET, log=True)

H3b – Régression linéaire – bloc complet (16 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       176.0             166.3                   -5.8                            48.7
      maison         2023       165.7             111.7                  -48.3                            30.7
 appartement         2024       208.6             148.0                  -41.0                            28.8
      maison         2024       203.3             129.2                  -57.4                            20.2


### 4.5 Variante : bloc complet + densité

In [10]:
lancer("Régression linéaire – bloc complet + densité", LinearRegression(), BLOC_COMPLET + DENSITE, log=True)

H3b – Régression linéaire – bloc complet + densité (17 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       175.4             166.3                   -5.5                            48.9
      maison         2023       163.0             111.7                  -45.9                            31.1
 appartement         2024       204.7             148.0                  -38.3                            30.5
      maison         2024       198.5             129.2                  -53.6                            20.8


'H3b – Régression linéaire – bloc complet + densité'

### 4.6 Lasso : quelles variables survivent
Le Lasso ramène à zéro les coefficients des variables qui n'apportent rien. `alpha` est réglé sur les plis temporels. Les valeurs testées sont petites : la cible est en log et les variables sont mises à l'échelle, les coefficients sont donc petits eux aussi.

In [11]:
LASSO = lancer("Lasso – bloc complet", Lasso(max_iter=10_000), BLOC_COMPLET, log=True,
               grille={"alpha": [0.0001, 0.001, 0.01, 0.1]})

H3b – Lasso – bloc complet (16 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       166.9             166.3                   -0.3                            49.0
      maison         2023       165.5             111.7                  -48.2                            30.7
 appartement         2024       194.4             148.0                  -31.4                            29.5
      maison         2024       200.3             129.2                  -55.0                            20.2


### 4.7 Gradient boosting : bloc complet (gabarit commun P3)
Gabarit de l'équipe (`gabarit_boosting.py`) : même modèle, même grille et même réglage pour toutes les hypothèses, ce qui rend les boostings comparables d'une hypothèse à l'autre. Le modèle apprend l'évolution du prix par rapport au prix N-1 et rend une prévision en €/m² : la cible jugée reste le prix. Un arbre qui prévoirait directement le prix rendrait la moyenne des prix de chaque feuille et perdrait la précision du prix N-1 de chaque sous-bassin. Pas de mise à l'échelle ni de log : un arbre n'en a pas besoin.

In [12]:
BOOSTING = lancer("Gradient boosting – bloc complet", BoostingEvolution(), BLOC_COMPLET)

H3b – Gradient boosting – bloc complet (16 variables)
type de bien  année jugée  MAE (€/m²)  MAE naïve (€/m²)  gain sur la naïve (%)  bassins où le modèle gagne (%)
 appartement         2023       182.0             166.3                   -9.4                            48.1
      maison         2023       172.0             111.7                  -54.0                            31.0
 appartement         2024       217.8             148.0                  -47.2                            27.1
      maison         2024       208.5             129.2                  -61.4                            20.9


## 5. Mesures et tests
Trois tableaux, du plus simple au plus exigeant :
1. **`res` : les erreurs.** MAE du modèle et de la naïve en €/m², gain sur la naïve en %, part des sous-bassins où le modèle gagne. C'est une description : un modèle peut avoir une MAE plus basse par hasard.
2. **`tst` : les tests contre la naïve.** Ils disent si l'écart avec la naïve est plus grand que le hasard.
   - **Écart de MAE avec la naïve** : erreur du modèle moins erreur de la naïve, en €/m² (négatif : le modèle fait mieux).
   - **IC 95 %** (bootstrap par bassin de vie) : la fourchette dans laquelle se trouve cet écart, en tenant compte du hasard. Si toute la fourchette est sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure ; si elle est toute au-dessus de 0, il fait vraiment moins bien.
   - **p-value de Wilcoxon** : le modèle fait-il mieux (ou moins bien) dans la plupart des sous-bassins, ou est-ce que cela se compense ? Avec plusieurs centaines de sous-bassins, elle est presque toujours très petite : on la lit avec la taille d'effet.
   - **Rang-bisériale** (taille d'effet) : de −1 (le modèle fait mieux partout) à +1 (il fait moins bien partout) ; autour de 0, pas de différence nette.
3. **`decision` : la règle de décision** (feuille de route 1.5). Une année est réussie si l'IC est entièrement sous 0, la rang-bisériale négative et le gain d'au moins 3 %. Un modèle est retenu s'il réussit la majorité des années jugées (ici, les deux).

In [13]:
res = pd.DataFrame(resultats)
res.round(2)

,modèle,type de bien,année jugée,MAE (€/m²),MAE en % du prix,MAE naïve (€/m²),gain sur la naïve (%),bassins où le modèle gagne (%)
0,H3b – Régression linéaire – prix N-1 seul,appartement,2023,185.46,7.07,166.33,-11.51,45.64
1,H3b – Régression linéaire – prix N-1 seul,maison,2023,170.00,7.76,111.71,-52.18,30.96
2,H3b – Régression linéaire – prix N-1 seul,appartement,2024,221.51,8.50,148.00,-49.67,26.97
3,H3b – Régression linéaire – prix N-1 seul,maison,2024,214.92,9.64,129.19,-66.36,18.24
4,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,171.28,6.70,166.33,-2.98,48.08
5,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,159.81,7.27,111.71,-43.06,31.69
6,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2024,207.64,8.06,148.00,-40.30,29.40
7,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2024,200.56,8.97,129.19,-55.25,20.21
8,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,175.22,6.81,166.33,-5.35,47.27
9,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,165.98,7.53,111.71,-48.58,30.73


In [14]:
tst = pd.DataFrame(tests)
tst.round(3)

,modèle,type de bien,année jugée,écart de MAE avec la naïve (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale,bassins de vie
0,H3b – Régression linéaire – prix N-1 seul,appartement,2023,19.140,-2.677,45.770,0.004,0.114,726
1,H3b – Régression linéaire – prix N-1 seul,maison,2023,58.291,50.608,65.476,0.000,0.471,1606
2,H3b – Régression linéaire – prix N-1 seul,appartement,2024,73.509,59.615,87.685,0.000,0.606,692
3,H3b – Régression linéaire – prix N-1 seul,maison,2024,85.732,80.270,91.580,0.000,0.778,1605
4,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,4.957,-8.278,17.728,0.127,0.060,726
5,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,48.103,41.050,54.511,0.000,0.429,1606
6,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2024,59.643,46.538,70.391,0.000,0.497,692
7,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2024,71.375,66.531,75.852,0.000,0.724,1605
8,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,8.891,-5.468,23.152,0.044,0.079,726
9,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,54.267,46.900,60.962,0.000,0.452,1606


### Règle de décision
Un modèle est retenu si, sur la majorité des années jugées : l'intervalle du bootstrap est entièrement sous 0, la rang-bisériale est négative et le gain sur la naïve atteint 3 %.

In [15]:
SEUIL_GAIN = 3      # % de gain sur la MAE de la naïve (décision d'équipe 1.6)

cles = ["modèle", "type de bien", "année jugée"]
regle = res[cles + ["gain sur la naïve (%)"]].merge(tst[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
regle["année réussie"] = ((regle["IC 95 % haut (€/m²)"] < 0) & (regle["rang-bisériale"] < 0)
                          & (regle["gain sur la naïve (%)"] >= SEUIL_GAIN))
decision = regle.groupby(["modèle", "type de bien"])["année réussie"].agg(["sum", "count"])
decision["retenu"] = decision["sum"] > decision["count"] / 2
decision.rename(columns={"sum": "années réussies", "count": "années jugées"})

années réussies  \
modèle                                             type de bien                    
H3b – Gradient boosting – bloc complet             appartement                 0   
                                                   maison                      0   
H3b – Lasso – bloc complet                         appartement                 0   
                                                   maison                      0   
H3b – Régression linéaire – bloc complet           appartement                 0   
                                                   maison                      0   
H3b – Régression linéaire – bloc complet + densité appartement                 0   
                                                   maison                      0   
H3b – Régression linéaire – prix N-1 + parc + v... appartement                 0   
                                                   maison                      0   
H3b – Régression linéaire – prix N-1 + parc + v... appartement                 0   
                                                   maison                      0   
H3b – Régression linéaire – prix N-1 seul          appartement                 0   
                                                   maison                      0   

                                                                 années jugées  \
modèle                                             type de bien                  
H3b – Gradient boosting – bloc complet             appartement               2   
                                                   maison                    2   
H3b – Lasso – bloc complet                         appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – bloc complet           appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – bloc complet + densité appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – prix N-1 + parc + v... appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – prix N-1 + parc + v... appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – prix N-1 seul          appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèle                                             type de bien          
H3b – Gradient boosting – bloc complet             appartement    False  
                                                   maison         False  
H3b – Lasso – bloc complet                         appartement    False  
                                                   maison         False  
H3b – Régression linéaire – bloc complet           appartement    False  
                                                   maison         False  
H3b – Régression linéaire – bloc complet + densité appartement    False  
                                                   maison         False  
H3b – Régression linéaire – prix N-1 + parc + v... appartement    False  
                                                   maison         False  
H3b – Régression linéaire – prix N-1 + parc + v... appartement    False  
                                                   maison         False  
H3b – Régression linéaire – prix N-1 seul          appartement    False  
                                                   maison         False

**Comment lire le tableau des tests (exemple : Lasso, appartements, 2024) :**
- **Écart de MAE avec la naïve = +46,4 €/m²** : le Lasso se trompe en moyenne de 46 €/m² de plus que la naïve.
- **IC 95 % = [+28,0 ; +60,2]** : toute la fourchette est au-dessus de 0 ; le Lasso fait vraiment moins bien que la naïve, ce n'est pas un effet du hasard.
- **p-value de Wilcoxon < 0,001 et rang-bisériale = 0,47** : le Lasso fait moins bien dans une nette majorité des sous-bassins (la rang-bisériale va de −1, le modèle fait mieux partout, à +1, il fait moins bien partout).

**Résultats :**
- **Aucun modèle n'est retenu** : aucune année réussie sur deux, pour les 7 modèles et les deux types de bien. Aucun modèle ne fait mieux que la naïve.
- **Dans la plupart des cas, les modèles font nettement moins bien que la naïve** : intervalle entièrement au-dessus de 0, rang-bisériale positive. L'écart est le plus fort pour les maisons en 2024 (de +69 à +86 €/m², rang-bisériale de 0,71 à 0,78) : le modèle fait moins bien dans presque tous les sous-bassins.
- **Une exception : les appartements en 2023.** Pour les six régressions (référence comprise) et pour le gradient boosting, l'intervalle contient 0 : on ne peut pas dire que le modèle fasse mieux ou moins bien que la naïve. Le Lasso y est à égalité presque parfaite (écart +0,5 €/m², IC [−9,2 ; +10,5], rang-bisériale 0,01).
- **Entre les modèles** (MAE, sans test) : ajouter le parc et la vacance au prix N-1 réduit l'erreur d'environ 14 €/m² pour chaque année et chaque type de bien (par exemple de 214,9 à 200,6 €/m² pour les maisons en 2024). Les ménages et les revenus ne la réduisent pas davantage ; la densité la réduit de 2 à 5 €/m². Le Lasso a l'erreur la plus basse pour les appartements ; le modèle « prix N-1 seul » et le gradient boosting ont les plus hautes.

**Ce que ces tests ne disent pas :** ils comparent chaque modèle à la naïve, pas au modèle « prix N-1 seul ». La baisse d'erreur apportée par le parc et la vacance par rapport à cette référence n'est donc pas encore testée.

### 5.2 Le bloc apporte-t-il quelque chose au prix N-1 seul ?
Contre la naïve, aucun modèle n'est retenu. La question propre à l'hypothèse est autre : les variables du logement et des ménages améliorent-elles la prévision **par rapport au modèle « prix N-1 seul »** ?

On applique exactement les mêmes mesures et la même règle de décision, en remplaçant la naïve par la prévision du modèle « prix N-1 seul ». Les fonctions `evaluer` et `tester` lisent la prévision de comparaison dans la colonne `prix_n1` : on leur donne une copie de l'année jugée où cette colonne contient la prévision de la référence.

Lecture : un gain positif et un intervalle du bootstrap entièrement sous 0 veulent dire que le modèle fait vraiment mieux que la référence.

In [16]:
resultats_ref, tests_ref = [], []
modeles_compares = [n for n in dict.fromkeys(k[0] for k in previsions) if n != REFERENCE]

for nom in modeles_compares:
    for annee, (app, ev) in FENETRES.items():
        ev_ref = ev.copy()
        ev_ref["prix_n1"] = previsions[(REFERENCE, annee)]     # la comparaison se fait avec la référence
        evaluer(nom, ev_ref, previsions[(nom, annee)], resultats_ref)
        tester(nom, ev_ref, previsions[(nom, annee)], tests_ref)

# Mêmes tableaux qu'en section 5, avec « référence » à la place de « naïve »
res_ref = pd.DataFrame(resultats_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
tst_ref = pd.DataFrame(tests_ref).rename(columns=lambda c: c.replace("naïve", "référence"))
display(res_ref[["modèle", "type de bien", "année jugée", "MAE (€/m²)", "MAE référence (€/m²)",
                 "gain sur la référence (%)", "bassins où le modèle gagne (%)"]].round(2))
display(tst_ref[["modèle", "type de bien", "année jugée", "écart de MAE avec la référence (€/m²)",
                 "IC 95 % bas (€/m²)", "IC 95 % haut (€/m²)", "p-value Wilcoxon", "rang-bisériale"]].round(3))

,modèle,type de bien,année jugée,MAE (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,171.28,185.46,7.65,59.12
1,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,159.81,170.00,5.99,66.29
2,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2024,207.64,221.51,6.26,60.75
3,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2024,200.56,214.92,6.68,67.51
4,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,175.22,185.46,5.53,56.68
5,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,165.98,170.00,2.37,58.60
6,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2024,211.31,221.51,4.61,58.32
7,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2024,204.75,214.92,4.73,63.06
8,H3b – Régression linéaire – bloc complet,appartement,2023,176.04,185.46,5.08,56.68
9,H3b – Régression linéaire – bloc complet,maison,2023,165.70,170.00,2.53,58.71


,modèle,type de bien,année jugée,écart de MAE avec la référence (€/m²),IC 95 % bas (€/m²),IC 95 % haut (€/m²),p-value Wilcoxon,rang-bisériale
0,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,-14.183,-32.802,-2.216,0.000,-0.201
1,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,-10.189,-13.209,-7.520,0.000,-0.403
2,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2024,-13.866,-36.049,1.332,0.000,-0.209
3,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2024,-14.357,-19.561,-10.343,0.000,-0.411
4,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2023,-10.249,-27.445,1.330,0.000,-0.138
5,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2023,-4.024,-7.186,-1.180,0.000,-0.170
6,H3b – Régression linéaire – prix N-1 + parc + ...,appartement,2024,-10.203,-30.448,4.224,0.000,-0.156
7,H3b – Régression linéaire – prix N-1 + parc + ...,maison,2024,-10.171,-15.485,-6.047,0.000,-0.298
8,H3b – Régression linéaire – bloc complet,appartement,2023,-9.429,-24.464,0.961,0.000,-0.147
9,H3b – Régression linéaire – bloc complet,maison,2023,-4.300,-7.422,-1.466,0.000,-0.180


#### Règle de décision contre la référence
Même règle qu'en section 5, avec le modèle « prix N-1 seul » à la place de la naïve : une année est réussie si l'intervalle du bootstrap est entièrement sous 0, la rang-bisériale négative et le gain sur la référence d'au moins 3 % ; un modèle est retenu s'il réussit les deux années jugées.

In [17]:
cles = ["modèle", "type de bien", "année jugée"]
regle_ref = res_ref[cles + ["gain sur la référence (%)"]].merge(
    tst_ref[cles + ["IC 95 % haut (€/m²)", "rang-bisériale"]], on=cles)
regle_ref["année réussie"] = ((regle_ref["IC 95 % haut (€/m²)"] < 0) & (regle_ref["rang-bisériale"] < 0)
                              & (regle_ref["gain sur la référence (%)"] >= SEUIL_GAIN))
decision_ref = regle_ref.groupby(["modèle", "type de bien"], sort=False)["année réussie"].agg(["sum", "count"])
decision_ref["retenu"] = decision_ref["sum"] > decision_ref["count"] / 2
decision_ref.rename(columns={"sum": "années réussies", "count": "années jugées"})

années réussies  \
modèle                                             type de bien                    
H3b – Régression linéaire – prix N-1 + parc + v... appartement                 1   
                                                   maison                      2   
H3b – Régression linéaire – prix N-1 + parc + v... appartement                 0   
                                                   maison                      1   
H3b – Régression linéaire – bloc complet           appartement                 1   
                                                   maison                      1   
H3b – Régression linéaire – bloc complet + densité appartement                 2   
                                                   maison                      2   
H3b – Lasso – bloc complet                         appartement                 2   
                                                   maison                      1   
H3b – Gradient boosting – bloc complet             appartement                 0   
                                                   maison                      0   

                                                                 années jugées  \
modèle                                             type de bien                  
H3b – Régression linéaire – prix N-1 + parc + v... appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – prix N-1 + parc + v... appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – bloc complet           appartement               2   
                                                   maison                    2   
H3b – Régression linéaire – bloc complet + densité appartement               2   
                                                   maison                    2   
H3b – Lasso – bloc complet                         appartement               2   
                                                   maison                    2   
H3b – Gradient boosting – bloc complet             appartement               2   
                                                   maison                    2   

                                                                 retenu  
modèle                                             type de bien          
H3b – Régression linéaire – prix N-1 + parc + v... appartement    False  
                                                   maison          True  
H3b – Régression linéaire – prix N-1 + parc + v... appartement    False  
                                                   maison         False  
H3b – Régression linéaire – bloc complet           appartement    False  
                                                   maison         False  
H3b – Régression linéaire – bloc complet + densité appartement     True  
                                                   maison          True  
H3b – Lasso – bloc complet                         appartement     True  
                                                   maison         False  
H3b – Gradient boosting – bloc complet             appartement    False  
                                                   maison         False

**Comment lire le tableau des tests (exemple : prix N-1 + parc + vacance, maisons, 2024) :** écart de −14,4 €/m² avec la référence, IC 95 % [−19,6 ; −10,3] entièrement sous 0, rang-bisériale −0,41, gain de 6,7 % : le modèle fait vraiment mieux que le prix N-1 seul, dans une nette majorité des sous-bassins (67,5 %). L'année est réussie.

**Résultats contre le modèle « prix N-1 seul » :**
- **Trois modèles sont retenus** (deux années réussies sur deux) :
  - **bloc complet + densité**, pour les maisons et pour les appartements : gain de 4 à 8 % selon l'année et le type de bien ;
  - **prix N-1 + parc + vacance**, pour les maisons : gain de 6 à 7 % ;
  - **Lasso sur le bloc complet**, pour les appartements : gain de 10 et 12 %, le plus fort de tous.
- **Le parc et la vacance portent l'essentiel de l'apport.** Ajouter les ménages fait perdre l'année 2023 pour les deux types de bien (gain des maisons ramené de 6,0 à 2,4 %, sous le seuil de 3 %). Les revenus ne regagnent rien. La densité rend le bloc retenu pour les deux types de bien.
- **Le gradient boosting fait à peu près jeu égal avec la référence** : gain de −1,2 à +3,0 %, aucune année réussie. Le cas le plus proche, les maisons en 2024 (IC entièrement sous 0, gain de 2,98 %), reste sous le seuil de 3 %.
- **Les gains restent modestes** (4 à 12 % de l'erreur de la référence) et aucun de ces modèles ne bat la naïve (section 5.1).
- **Prudence :** plusieurs variantes sont comparées sur les mêmes années jugées ; celle qui ressort peut en partie en profiter. Le test final sur 2025, fait une seule fois sur le modèle retenu, le confirmera.

## 6. Interprétation
Cinq lectures des résultats : le contexte du marché sur les années jugées, les variables qui portent la prévision, la qualité des données, la taille du marché, et l'effet de seuil des logements anciens.

### 6.1 Contexte : le marché s'est-il retourné ?
Les modèles apprennent sur 2020-2022, puis 2023, et sont jugés sur 2023 et 2024. Pour lire leurs résultats, on mesure l'évolution du prix d'une année sur l'autre (prix / prix N-1 − 1), en moyenne et en médiane, par année et par type de bien, jusqu'à 2024. L'année 2025, réservée au test final, n'est pas regardée.

In [18]:
jusqua_2024 = table[table["annee"] <= commun.ANNEE_VALIDATION]
evolution = (jusqua_2024[CIBLE] / jusqua_2024["prix_n1"] - 1) * 100
retournement = (evolution.groupby([jusqua_2024["annee"], jusqua_2024.index.get_level_values("type_bien")])
                .agg(["mean", "median"]).round(1)
                .rename(columns={"mean": "évolution moyenne (%)", "median": "évolution médiane (%)"}))
retournement.unstack()

évolution moyenne (%)        évolution médiane (%)       
type_bien           appartement maison           appartement maison
annee                                                              
2017                        2.2    1.8                   2.1    2.0
2018                        1.4    2.1                   1.5    2.1
2019                        3.2    2.3                   3.0    2.3
2020                        5.2    4.7                   5.1    4.7
2021                        8.0    7.9                   7.5    7.7
2022                        8.8    9.6                   8.2    9.1
2023                        2.9    1.3                   2.9    0.8
2024                       -0.8   -2.4                  -1.5   -2.9

**Comment lire le tableau (exemple : maisons, 2022) :** en moyenne, le prix médian au m² d'un sous-bassin a augmenté de 9,6 % par rapport à 2021.

**Constat :**
- **2017-2019 : hausse lente**, de +1,4 à +3,2 % par an.
- **2020-2022 : forte hausse**, de +4,7 à +9,6 % par an. Ce sont les années d'apprentissage.
- **2023 : net ralentissement** (+2,9 % pour les appartements, +1,3 % pour les maisons), **2024 : baisse** (−0,8 % et −2,4 %). Ce sont les années jugées.
- Les modèles apprennent donc sur des années de forte hausse et sont jugés sur un marché qui se retourne. Un modèle qui prolonge la hausse apprise surestime le prix ; la naïve (prix inchangé) n'est pas touchée. C'est ce qui explique que tous les modèles, y compris « prix N-1 seul », perdent contre la naïve (section 5.1), surtout pour les maisons en 2024.
- Les variables du bloc sont des photos fixes (parc 2022, vacance 2023, revenus 2021) : elles ne changent pas d'une année à l'autre et ne peuvent pas annoncer un retournement.

### 6.2 Importance des variables
Importance par permutation, sur l'année jugée : on mélange au hasard les valeurs d'une variable entre les sous-bassins, 10 fois, et on mesure de combien l'erreur du modèle change. Modèle étudié : le Lasso sur le bloc complet, pour les maisons en 2024, le cas le plus touché par le retournement.

In [19]:
choix = (LASSO, 2024, "maison")        # modèle, année jugée, type de bien étudiés
m, X_ev, y_ev = ajustes[choix]
imp = permutation_importance(m, X_ev, y_ev, scoring="neg_mean_absolute_error",
                             n_repeats=10, random_state=commun.GRAINE)
pd.DataFrame({"hausse de la MAE (€/m²)": imp.importances_mean,
              "écart-type (€/m²)": imp.importances_std},
             index=X_ev.columns).sort_values("hausse de la MAE (€/m²)", ascending=False).round(1)

,hausse de la MAE (€/m²),écart-type (€/m²)
prix_n1,956.9,17.0
part_rsec,1.8,0.3
taux_personnes_seules_pct_men,0.4,0.1
part_couples_sans_enfants_pct_fam,0.3,0.0
part_familles_nombreuses_pct_fam,0.0,0.0
part_familles_monoparentales_pct_fam,0.0,0.0
log_nb_logements,0.0,0.0
part_sans_voiture,-0.1,0.0
part_apparts,-1.1,0.0
taux_vacance_complete,-2.0,0.2


**Comment lire le tableau (exemple : `taux_vacance`, −7,6 €/m², écart-type 0,6) :** quand on mélange au hasard la vacance du recensement entre les sous-bassins, l'erreur du Lasso *baisse* de 7,6 €/m² en moyenne sur 10 mélanges. L'écart-type est petit (0,6) : ce n'est pas un effet du hasard. Sur cette année, la variable dessert le modèle.
Règle : une variable compte (dans un sens ou dans l'autre) si sa valeur dépasse nettement deux écarts-types ; 0 veut dire que le Lasso ne s'en sert pas (coefficient ramené à zéro).

**Constat (Lasso, bloc complet, maisons, 2024) :**
- **Le prix N-1 porte presque tout** : le brouiller fait monter l'erreur de 957 €/m².
- **Le reste du bloc n'aide pas en 2024**, et plusieurs variables desservent le modèle : vacance du recensement (−7,6 €/m²), logements d'avant 1946 et logements récents (−5,1 et −5,0), niveau de vie (−4,9), ancienneté d'emménagement (−3,5). Seules les résidences secondaires aident un peu (+1,8).
- **Le Lasso écarte trois variables** (importance nulle) : familles nombreuses, familles monoparentales, taille du parc.

**Hypothèse, vérifiée pour les logements d'avant 1946 (section 6.5) :** ces variables distinguent des territoires qui n'évoluent pas au même rythme. Les territoires où les logements anciens sont rares ont plus monté que les autres jusqu'en 2022, puis ont plus baissé en 2024. Le modèle a appris le premier mouvement ; en 2024, il pousse la prévision dans le mauvais sens. Pour les autres variables (vacance, niveau de vie, logements récents), le même mécanisme est probable, mais n'a pas été vérifié.

**À rapprocher de la section 5.2 :** le bloc réduit l'erreur par rapport au prix N-1 seul, alors que chaque variable, prise une à une, la fait monter ici. Les deux mesures ne répondent pas à la même question : la section 5.2 compare deux modèles entraînés différemment, la permutation brouille une variable dans un modèle déjà entraîné, les autres variables (liées entre elles) restant en place.

### 6.3 Analyse des erreurs selon le contrôle qualité
L'erreur est-elle plus grande là où les données sont estimées ou complétées ? Pour chaque variable de contrôle qualité : groupe « part nulle », puis quartiles parmi les parts non nulles. Par groupe : erreur moyenne en €/m² et en % du prix, et erreur moyenne signée. Rho de Spearman entre la variable et l'erreur, en €/m² et en %. Même modèle qu'en 6.2.

In [20]:
CONTROLE = ["part_indicateurs_estimes", "part_logements_taux_impute", "part_menages_couverts"]

m, X_ev, y_ev = ajustes[choix]
erreur = pd.Series(m.predict(X_ev), index=X_ev.index) - y_ev       # signée : positive = prix surestimé
erreur_pct = erreur.abs() / y_ev * 100                             # erreur absolue en % du prix réel
controle = table.loc[X_ev.index, CONTROLE]

for v in CONTROLE:
    x = controle[v]
    groupe = pd.Series("part nulle", index=x.index)
    non_nul = x > 0
    if non_nul.sum() >= 4:
        groupe[non_nul] = pd.qcut(x[non_nul], 4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop").astype(str)
    tableau = pd.DataFrame({"MAE (€/m²)": erreur.abs(), "erreur en % du prix": erreur_pct,
                            "erreur signée (€/m²)": erreur}).groupby(groupe).mean()
    tableau["lignes"] = groupe.value_counts()
    rho, p = stats.spearmanr(x, erreur.abs())
    rho_pct, _ = stats.spearmanr(x, erreur_pct)
    print(f"\n{v} : rho avec l'erreur en €/m² = {rho:.2f} (p = {p:.3f}) | rho avec l'erreur en % = {rho_pct:.2f}")
    print(tableau.round(1).to_string())


part_indicateurs_estimes : rho avec l'erreur en €/m² = -0.18 (p = 0.000) | rho avec l'erreur en % = -0.05
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               258.3                  9.2                 244.6     439
Q2               178.4                  8.9                 154.8     439
Q3               174.5                  8.8                 151.7     438
Q4               183.0                  9.0                 156.4     439
part nulle       342.4                  8.5                 288.0      21

part_logements_taux_impute : rho avec l'erreur en €/m² = -0.35 (p = 0.000) | rho avec l'erreur en % = -0.11
            MAE (€/m²)  erreur en % du prix  erreur signée (€/m²)  lignes
Q1               221.3                  9.0                 212.9     346
Q2               169.5                  8.5                 140.5     345
Q3               151.0                  8.5                 124.7     345
Q4               137.7                  9.0 

**Comment lire les tableaux (exemple : `part_logements_taux_impute`) :** les sous-bassins sont répartis en groupes selon la part de leurs logements dont le taux de vacance a été complété par l'intercommunalité : « part nulle » (aucun), puis quatre quartiles parmi les autres (Q1 = parts les plus faibles, Q4 = les plus fortes). Pour chaque groupe :
- **MAE** : l'erreur moyenne du modèle, en €/m² ;
- **erreur en % du prix** : la même erreur rapportée au prix réel du sous-bassin ;
- **erreur signée** : positive quand le modèle surestime le prix.

Le **rho de Spearman** mesure le lien entre la variable de contrôle et l'erreur : proche de 0, pas de lien. Ici, −0,35 avec l'erreur en €/m², mais −0,11 seulement avec l'erreur en % : le lien vient surtout du niveau des prix.

**Constat (Lasso, bloc complet, maisons, 2024) :**
- **En €/m², l'erreur est plus grande là où les données sont les plus complètes** : 306 €/m² quand aucun taux de vacance n'est complété, contre 138 €/m² dans le quartile le plus complété ; 342 €/m² pour les 21 sous-bassins sans indicateur de délinquance estimé. Ce sont les territoires les plus grands et les plus urbains, où les prix sont les plus élevés.
- **En % du prix, l'erreur est presque la même partout** : de 8,5 à 9,9 % selon les groupes, pour les trois variables de contrôle (rho de −0,11 à 0,01). L'écart en €/m² vient du niveau des prix, pas de la qualité des données.
- **Le modèle surestime le prix dans tous les groupes** (erreur signée de +111 à +288 €/m²), ce qui est cohérent avec le retournement du marché en 2024 (section 6.1).

**Conclusion :** l'erreur n'est pas concentrée dans les territoires aux données estimées ou complétées. La qualité des données n'est pas une limite pour ce modèle.

### 6.4 Le gain dépend-il de la taille du marché ?
Hypothèse : le parc de logements donne un prix attendu stable, qui aide surtout là où le prix médian repose sur peu de ventes (petits territoires, prix plus irréguliers d'une année sur l'autre). Pour le modèle retenu pour les deux types de bien (bloc complet + densité, section 5.2), on compare son erreur à celle du modèle « prix N-1 seul », par quartile du nombre de ventes de l'année jugée (Q1 = les marchés les plus petits). Le nombre de ventes ne sert ici qu'à classer les sous-bassins : il n'entre dans aucun modèle.

In [21]:
MODELE_ETUDIE = f"{HYPOTHESE} – Régression linéaire – bloc complet + densité"

lignes = []
for annee, (app, ev) in FENETRES.items():
    for type_bien, e in ev.groupby("type_bien"):
        err_modele = (previsions[(MODELE_ETUDIE, annee)].loc[e.index] - e[CIBLE]).abs()
        err_ref = (previsions[(REFERENCE, annee)].loc[e.index] - e[CIBLE]).abs()
        quartile = pd.qcut(e["n_ventes"], 4, labels=["Q1 (petits)", "Q2", "Q3", "Q4 (grands)"])
        for q, idx in e.groupby(quartile, observed=True).groups.items():
            lignes.append({"type de bien": type_bien, "année jugée": annee, "quartile de ventes": q,
                           "ventes (min – max)": f"{int(e.loc[idx, 'n_ventes'].min())} – {int(e.loc[idx, 'n_ventes'].max())}",
                           "MAE modèle (€/m²)": err_modele.loc[idx].mean(),
                           "MAE référence (€/m²)": err_ref.loc[idx].mean(),
                           "gain sur la référence (%)": (1 - err_modele.loc[idx].mean() / err_ref.loc[idx].mean()) * 100,
                           "bassins où le modèle gagne (%)": (err_modele.loc[idx] < err_ref.loc[idx]).mean() * 100})
pd.DataFrame(lignes).round(1)

,type de bien,année jugée,quartile de ventes,ventes (min – max),MAE modèle (€/m²),MAE référence (€/m²),gain sur la référence (%),bassins où le modèle gagne (%)
0,appartement,2023,Q1 (petits),20 – 42,181.1,190.2,4.8,58.1
1,appartement,2023,Q2,43 – 88,150.7,167.4,10.0,62.1
2,appartement,2023,Q3,89 – 256,159.8,154.2,-3.6,54.4
3,appartement,2023,Q4 (grands),258 – 52899,209.9,229.9,8.7,56.7
4,maison,2023,Q1 (petits),20 – 96,201.7,205.7,2.0,60.1
5,maison,2023,Q2,97 – 161,154.0,162.6,5.3,60.5
6,maison,2023,Q3,162 – 274,148.0,156.5,5.4,65.4
7,maison,2023,Q4 (grands),275 – 21884,147.5,154.6,4.6,60.4
8,appartement,2024,Q1 (petits),20 – 40,183.7,197.2,6.8,61.5
9,appartement,2024,Q2,41 – 85,161.0,184.2,12.6,64.9


**Comment lire le tableau (exemple : maisons, 2024, Q1) :** les sous-bassins de 20 à 90 ventes de maisons (le quart des marchés les plus petits). Le modèle s'y trompe en moyenne de 232,8 €/m², contre 250,5 €/m² pour le modèle « prix N-1 seul » : 7,1 % d'erreur en moins, et il fait mieux dans 69 % de ces sous-bassins.

**Constat :**
- **Le gain sur la référence est présent dans presque tous les groupes** : positif dans 15 cas sur 16 (de +2,0 à +12,6 %), et le modèle fait mieux que la référence dans 54 à 69 % des sous-bassins de chaque groupe. Seule exception : les appartements de taille moyenne (Q3) en 2023 (−3,6 %).
- **Le gain n'est pas plus fort dans les petits marchés.** Pour les maisons, il est le plus faible dans le quartile des plus petits en 2023 (+2,0 %), et du même ordre que les autres en 2024 (+7,1 %). Pour les appartements, il est le plus fort dans le deuxième quartile (+10,0 et +12,6 %), pas dans le premier.
- **L'hypothèse d'un apport concentré sur les petits territoires n'est donc pas confirmée** : le parc de logements améliore la prévision dans les marchés de toutes tailles.
- Ces écarts entre quartiles ne sont pas testés (pas d'intervalle de confiance par groupe) : ils décrivent une tendance, sans plus.

### 6.5 Effet de seuil des logements d'avant 1946 (maisons)
Dans l'audit de la table, la forêt aléatoire donnait un effet de seuil : peu de logements d'avant 1946 faisaient nettement monter le prix des maisons. Ici, le prix N-1 porte déjà le niveau du prix ; ce que la part de logements anciens peut encore apporter, c'est une différence d'évolution. Par décile de la part de logements d'avant 1946 (D1 = les moins anciens), on calcule le prix médian et l'évolution moyenne du prix (prix / prix N-1 − 1), année par année jusqu'à 2024. Cette lecture sert aussi à vérifier l'hypothèse de la section 6.2.

In [22]:
maisons = table[(table.index.get_level_values("type_bien") == "maison") & (table["annee"] <= commun.ANNEE_VALIDATION)]
decile = pd.qcut(maisons["part_avant1946"], 10, labels=[f"D{i}" for i in range(1, 11)])
evolution_maisons = (maisons[CIBLE] / maisons["prix_n1"] - 1) * 100

seuil = evolution_maisons.groupby([decile, maisons["annee"]], observed=True).mean().unstack().round(1)
seuil.columns = [f"évolution {a} (%)" for a in seuil.columns]
seuil.insert(0, "part d'avant 1946 (%)", maisons.groupby(decile, observed=True)["part_avant1946"]
             .agg(lambda s: f"{s.min():.0f} – {s.max():.0f}"))
seuil.insert(1, "prix médian (€/m²)", maisons.groupby(decile, observed=True)[CIBLE].median().round(0))
seuil

,part d'avant 1946 (%),prix médian (€/m²),évolution 2017 (%),évolution 2018 (%),évolution 2019 (%),évolution 2020 (%),évolution 2021 (%),évolution 2022 (%),évolution 2023 (%),évolution 2024 (%)
part_avant1946,,,,,,,,,,
D1,2 – 11,3115.0,3.1,3.6,2.8,6.1,8.6,10.3,1.9,-4.1
D2,11 – 14,2587.0,2.3,3.4,3.5,5.2,10.0,9.8,1.5,-4.0
D3,14 – 18,2333.0,2.6,2.6,3.2,6.0,8.3,10.3,1.3,-3.3
D4,18 – 21,2017.0,2.9,2.3,2.4,4.7,7.9,9.5,0.9,-2.6
D5,21 – 24,1806.0,1.7,2.7,2.1,4.3,7.8,9.3,0.9,-2.5
D6,24 – 27,1641.0,2.4,1.2,2.5,4.8,6.8,9.1,1.4,-2.0
D7,27 – 30,1466.0,1.9,1.1,1.8,4.5,8.7,9.6,0.9,-1.5
D8,30 – 34,1361.0,0.0,2.3,1.1,3.8,7.6,9.1,1.4,-2.0
D9,34 – 40,1204.0,0.8,1.2,2.2,4.1,7.1,9.4,1.3,-1.1


**Comment lire le tableau (exemple : D1) :** le dixième des lignes où les maisons d'avant 1946 sont les plus rares (2 à 11 % du parc). Le prix médian y est de 3 115 €/m² ; en moyenne, le prix y a monté de 10,3 % en 2022 et baissé de 4,1 % en 2024.

**Constat :**
- **Niveau du prix : un gradient régulier plutôt qu'un seuil.** Le prix médian baisse à chaque décile, de 3 115 €/m² (D1) à 978 €/m² (D10). Le seul saut un peu plus marqué se trouve entre D1 et D2 (−528 €/m², autour de 11 % de logements anciens) ; ensuite, la baisse est de 100 à 300 €/m² par décile. Avec le prix N-1 dans le modèle, ce niveau est déjà pris en compte.
- **Évolution : les territoires les moins anciens bougent le plus, dans les deux sens.**
  - 2017-2019 : ils montent plus (+2,3 à +3,6 % par an pour D1 à D3, contre 0 à +2,2 % pour D8 à D10).
  - 2020-2022 : tous montent fortement ; les moins anciens un peu plus en 2021 (+8,3 à +10,0 % contre +6,4 à +7,1 %), tous de façon semblable en 2022 (+9 à +10 %).
  - 2024 : ce sont eux qui baissent le plus (−4,1 % pour D1, −0,6 % pour D10). L'écart d'évolution entre déciles s'inverse : les territoires qui montaient le plus baissent le plus.
- **Conséquence pour le modèle :** sur les années d'apprentissage, une faible part de logements anciens signale une hausse plus forte. En 2024, elle signale au contraire une baisse plus forte. La variable pousse la prévision dans le mauvais sens, ce qui explique qu'elle desserve le Lasso en 2024 (section 6.2).
- Ces écarts sont des moyennes par décile, sans test ; ils décrivent une tendance, pas une cause.

## 7. Conclusion

**Question :** les caractéristiques du parc de logements, de la vacance, des ménages et des revenus améliorent-elles la prévision du prix médian au m², par rapport au prix de l'année précédente ? Le parc aide-t-il surtout dans les petits territoires ? Les revenus disent-ils la même chose que le parc ?

**1. Contre la naïve (prix N = prix N-1) : aucun modèle n'est retenu.**
- Aucun des 7 modèles ne réussit une seule des deux années jugées. Dans presque tous les cas, ils font nettement moins bien que la naïve (intervalle du bootstrap entièrement au-dessus de 0), surtout pour les maisons en 2024 (+69 à +86 €/m²).
- Seule exception : les appartements en 2023, où les régressions sont à égalité avec la naïve.
- **Cause : le retournement du marché** (section 6.1). Les modèles apprennent sur 2020-2022, des années de forte hausse (+5 à +10 % par an), et sont jugés sur 2023 (+1 à +3 %) et 2024 (−1 à −3 %). Ils prolongent la hausse apprise et surestiment le prix partout. Le modèle « prix N-1 seul » perd déjà contre la naïve : le problème ne vient pas du bloc. Les variables du bloc sont des photos fixes et ne peuvent pas annoncer un retournement.

**2. Contre le modèle « prix N-1 seul » : le bloc apporte une amélioration réelle mais modeste.**
- **Trois modèles sont retenus** (deux années réussies sur deux) :
  - bloc complet + densité, pour les maisons et les appartements (gain de 4 à 8 %) ;
  - prix N-1 + parc + vacance, pour les maisons (gain de 6 à 7 %) ;
  - Lasso sur le bloc complet, pour les appartements (gain de 10 et 12 %).
- **Le parc et la vacance portent l'essentiel de l'apport.** Les ménages n'apportent rien de plus : les ajouter fait perdre l'année 2023 pour les deux types de bien. **Les revenus disent la même chose que le parc** : ils ne réduisent pas l'erreur davantage. La densité rend le bloc retenu pour les deux types de bien.
- Le gradient boosting (gabarit commun) ne fait pas mieux que la référence : gain de −1,2 à +3,0 %, aucune année réussie.

**3. Le gain ne se concentre pas dans les petits territoires** (section 6.4). Il est présent dans les marchés de toutes tailles (positif dans 15 groupes sur 16), sans être plus fort dans le quart des marchés les plus petits. L'hypothèse d'un apport surtout utile aux petits territoires n'est pas confirmée.

**4. Les logements d'avant 1946 : un gradient plutôt qu'un seuil** (section 6.5). Le prix des maisons baisse régulièrement avec la part de logements anciens, sans seuil net. Les territoires où ces logements sont rares ont plus monté jusqu'en 2022, puis plus baissé en 2024 : le modèle a appris le premier mouvement, ce qui explique que la variable le desserve en 2024 (section 6.2).

**5. Qualité des données :** l'erreur n'est pas concentrée dans les territoires aux données estimées ou complétées ; rapportée au prix, elle est de 8,5 à 9,9 % dans tous les groupes (section 6.3).

**Bilan :** le bloc logement améliore la prévision par rapport au prix N-1 seul, principalement par le parc et la vacance, mais ne suffit pas à battre la naïve dans un marché qui se retourne. Pour prévoir un retournement, il faut des variables qui changent d'une année à l'autre (volume de ventes, conditions de crédit, capacité d'achat), que les photos fixes du territoire ne remplacent pas.

**Limites :**
- Plusieurs variantes ont été comparées sur les mêmes années jugées : celle qui ressort a pu en profiter en partie. Le test final sur 2025, fait une seule fois sur le modèle retenu, le confirmera.
- Les écarts entre quartiles (6.4), entre déciles (6.5) et l'importance des variables (6.2, une seule année et un seul type de bien) décrivent des tendances, sans test.